# BERT-base: H, HT, T, H-nat, T-nat (seeds 1-2)
Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.
Runs are saved to Google Drive in `MyDrive/colab/runs`. If the session dies, Run all again: finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, zipfile, torch
DRIVE = '/content/drive/MyDrive/colab'
OUT = f'{DRIVE}/runs'
os.makedirs(OUT, exist_ok=True)
if not os.path.exists('/content/p6/data/train_student.json'):
    zipfile.ZipFile(f'{DRIVE}/p6_colab.zip').extractall('/content/p6')
os.chdir('/content/p6')
sys.path.insert(0, '/content/p6')
!pip -q install ir-measures
print(torch.cuda.get_device_name(0))

In [ ]:
import json
from src.student import run_model, natural_subset, run_nat
train = json.load(open('data/train_student.json'))
train_nat = natural_subset(train)
evals = {s: json.load(open(f'data/eval_{s}.json')) for s in ['val', 'dl19', 'dl20']}
print(len(train), len(train_nat))
for seed in [1, 2]:
    run_model('bert-base-uncased', 'base', 2e-5, seed, train, evals, out_dir=OUT)
    run_nat('bert-base-uncased', 'base', 2e-5, seed, train_nat, evals, out_dir=OUT)
print('done')

When it's done, run the cell below to download `base_runs.zip` and unzip it into `NLP2/results/runs/`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/base_runs', 'zip', OUT)
files.download('/content/base_runs.zip')